In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import time
from datetime import datetime
import mujoco

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from gymnasium.envs.mujoco.humanoid_v5 import HumanoidEnv
from gymnasium.envs.registration import register
from IPython.display import HTML
from IPython.display import Video
from IPython.display import display, clear_output
from PIL import Image as PILImage

from stable_baselines3 import SAC, TD3
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import CheckpointCallback

import glob

import os
#os.environ["MUJOCO_GL"] = "glfw"

pd.set_option('display.max_columns', None)

<p>
  <img src="Humanoid.png" alt="Humanoid" width="800">
</p>

In [2]:
import custom_humanoid

env = gym.make("CustomHumanoid-v0")
obs, info = env.reset()

init
left knee:  17  right knee:  13


# Evaluacion

In [3]:
def evaluate_humanoid(env, model, n_episodes=10):
    results = []
    for i in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_len, ep_reward = 0, 0
        log = []
        
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            # 🎯 Métricas
            ep_len += 1
            ep_reward += reward
            
            data = env.unwrapped.data

            results.append({
                "Episode": i,
                "Ep_len" : ep_len,
                "Ep_reward" : ep_reward,
                "Action" : action,
                "Info": info,
                "Reward" : reward,
                "Qpos" : data.qpos,
                "Qvel" : data.qvel,
                "Obs" : obs,
                "End" : done
            })
    
    # 📈 Resumen estadístico
    return results

In [4]:
def df_expanded (df, col):
    df_expanded = df[col].apply(pd.Series).copy()
    
    if col == "Info":
        df_result = pd.DataFrame(df_expanded.values , columns=("Info_" + df_expanded.columns))
    elif col == "Qpos":
        qpos_names =['root_x', 'root_y', 'root_z', 'root_qw', 'root_qx', 'root_qy', 'root_qz', 'abdomen_z', 'abdomen_y', 'abdomen_x',
                        'right_hip_x', 'right_hip_z', 'right_hip_y', 'right_knee', 'left_hip_x', 'left_hip_z', 'left_hip_y',
                        'left_knee', 'right_shoulder1', 'right_shoulder2', 'right_elbow', 'left_shoulder1', 'left_shoulder2', 'left_elbow']
        qpos_names = [col + "_" + x for x in qpos_names]
        df_result = pd.DataFrame(df_expanded.values , columns=(qpos_names))
    elif col == "Qvel" :
        qvel_names =['root_vx', 'root_vy', 'root_vz', 'root_wx', 'root_wy', 'root_wz', 'abdomen_z_vel', 'abdomen_y_vel', 'abdomen_x_vel',
                        'right_hip_x_vel', 'right_hip_z_vel', 'right_hip_y_vel', 'right_knee_vel', 'left_hip_x_vel', 'left_hip_z_vel',
                        'left_hip_y_vel', 'left_knee_vel', 'right_shoulder1_vel', 'right_shoulder2_vel', 'right_elbow_vel', 'left_shoulder1_vel',
                        'left_shoulder2_vel', 'left_elbow_vel']
        qvel_names = [col + "_" + x for x in qvel_names]
        df_result = pd.DataFrame(df_expanded.values , columns=(qvel_names))
    elif col == "Action" :
        actions =['abdomen_y', 'abdomen_z', 'abdomen_x', 'right_hip_x', 'right_hip_z', 'right_hip_y', 'right_knee', 'left_hip_x', 'left_hip_z',
                    'left_hip_y', 'left_knee', 'right_shoulder1', 'right_shoulder2', 'right_elbow', 'left_shoulder1', 'left_shoulder2',
                    'left_elbow']
        actions = [col + "_" + x for x in actions]
        df_result = pd.DataFrame(df_expanded.values , columns=(actions))
    else:
        df_expanded.columns = [f"{col}_{i+1}" for i in range(df_expanded.shape[1])]
        df_result = pd.DataFrame(df_expanded)
    
    return df_result

In [5]:
def ArreglaryGuardar_df (df, nombre):
    path = "./metricas/"
    
    # Arreglos para los df, que todos tengan su columna
    df_info = df_expanded (df, "Info")
    df_qpos = df_expanded (df, "Qpos")
    df_qvel = df_expanded (df, "Qvel")
    df_action = df_expanded (df, "Action")

    # juntar las columnas extraidas en un df
    df_final = pd.concat([df, df_info], axis=1)
    df_final = pd.concat([df_final, df_qpos] , axis=1)
    df_final = pd.concat([df_final, df_qvel] , axis=1)
    df_final = pd.concat([df_final, df_action], axis=1)

    # hay alguna columna de los resultados que aun necesita ser expandida
    df_tendon_length = df_expanded (df_final, "Info_tendon_length")
    df_tendon_vel = df_expanded (df_final, "Info_tendon_velocity")

    df_final = pd.concat([df_final, df_tendon_length] , axis=1)
    df_final = pd.concat([df_final, df_tendon_vel] , axis=1)

    # guardo las OBS por si llegan a ser necesarias
    df_obs = df_expanded (df, "Obs")

    df_final.drop(columns=["Info", "Qpos", "Qvel", "Action", "Obs", "Info_tendon_length", "Info_tendon_velocity"] , inplace= True)  
    
    df.to_parquet( path + nombre + ".parquet", engine="pyarrow", compression="zstd") 
    df.to_excel( path + nombre + '.xlsx', index=False)

## Evaluate analisis

In [6]:
analisis = False

base_sac = './sac_models'
archivos_sac = os.listdir(base_sac)
print(archivos_sac)

model = SAC.load(base_sac + '/' + archivos_sac[0])

['sac_humanoid_final_1000000_20260430.zip', 'sac_humanoid_final_1000000_20260515_velocity_2_g0_95t0_005.zip', 'sac_humanoid_final_20000000_20260501.zip', 'sac_humanoid_final_20000000_20260505_velocity.zip', 'sac_humanoid_final_20000000_20260511_basic.zip', 'sac_humanoid_final_20000000_20260512_velocity.zip', 'sac_humanoid_final_20000000_20260516_knees_8_g95t5.zip', 'sac_humanoid_final_2000000_20260505_velocity.zip', 'sac_humanoid_final_30000000_20260520_knees_8_g95t5.zip', 'sac_humanoid_final_30000000_20260521_knees_8_g95t5.zip', 'sac_humanoid_final_30000000_20260525_knees_8_g90t5.zip', 'sac_humanoid_final_40000000_20260518_knees_8_g95t5.zip']


In [7]:
env = gym.make("Humanoid-v5")
env.reset()

action, _ = model.predict(obs, deterministic=True)
obs, reward, terminated, truncated, info = env.step(action)
done = terminated or truncated

model_env = env.unwrapped.model

if analisis == True:
    joint_names = [model_env.joint(i).name for i in range(model_env.njnt)]
    print("joint len: ", len(joint_names), "joint array: ", joint_names)

    body_names = [model_env.body(i).name for i in range(model_env.nbody)]
    print("body_names len: ", len(body_names), "body_names array: ",body_names)

    geom_names = [model_env.geom(i).name for i in range(model_env.ngeom)]
    print("geom_names len: ", len(geom_names), "geom_names array: ",geom_names)

    actuator_names = [model_env.actuator(i).name for i in range(model_env.nu)]
    print("actuator_names len: ", len(actuator_names), "actuator_names array: ", actuator_names)

    sensor_names = [model_env.sensor(i).name for i in range(model_env.nsensor)]
    print("sensor_names len: ", len(sensor_names), "sensor_names array: ",sensor_names)

    print("Rewards: ", reward)

    print("Info: ", info)  
    print("Action: ", action) 
                   
                
    data = env.unwrapped.data
    print("len qpos: ", len(data.qpos),", data qpos: ", data.qpos)
    print("len qvel: ", len(data.qvel),", data qvel: ", data.qvel)
    print("len cinert: ", len(data.cinert),", data cinert: ", data.cinert)
    print("len cvel: ", len(data.cvel),", data cvel: ", data.cvel) 
    print("len qfrc_actuator: ", len(data.qfrc_actuator),", data qfrc_actuator: ", data.qfrc_actuator) 
    print("len cfrc_ext: ", len(data.cfrc_ext),", data cfrc_ext: ", data.cfrc_ext)
    #print("len obs: ", len(obs), " obs: ", obs)

In [8]:
if analisis == True:
    qpos_names = []

    for i in range(model_env.njnt):
        name = model_env.joint(i).name
        start = model_env.jnt_qposadr[i]

        if i < model_env.njnt - 1:
            end = model_env.jnt_qposadr[i + 1]
        else:
            end = model_env.nq  # tamaño total de qpos

        dim = end - start

        # ahora sí puedes usar dim
        if dim == 7:
            qpos_names += [
                "root_x", "root_y", "root_z",
                "root_qw", "root_qx", "root_qy", "root_qz"
            ]
        elif dim == 1:
            qpos_names.append(name)
        else:
            for d in range(dim):
                qpos_names.append(f"{name}_{d}")

    qpos_names

In [9]:
if analisis == True:
    qvel_names = []

    for i in range(model_env.njnt):
        name = model_env.joint(i).name
        start = model_env.jnt_dofadr[i]

        if i < model_env.njnt - 1:
            end = model_env.jnt_dofadr[i + 1]
        else:
            end = model_env.nv  # total DOFs

        dim = end - start
        jtype = model_env.jnt_type[i]

        if jtype == 0:  # free joint (root)
            qvel_names += [
                "root_vx", "root_vy", "root_vz",
                "root_wx", "root_wy", "root_wz"
            ]

        elif jtype == 1:  # ball joint (3 DOF)
            qvel_names += [f"{name}_wx", f"{name}_wy", f"{name}_wz"]

        else:  # hinge / slide (1 DOF)
            qvel_names.append(f"{name}_vel")
            
    qvel_names

In [10]:
actuator_names = [model_env.actuator(i).name for i in range(model_env.nu)]

## Evaluate execution

In [11]:
# Ejecutar evaluación en distintos checkpoints
base_sac = './sac_models'
archivos_sac = os.listdir(base_sac)

print(archivos_sac)


metrics_log = []
for step, model_path in enumerate(archivos_sac):
    print("Cargando: ", model_path)
    print("step:", step)
    model = SAC.load(base_sac + '/' + model_path)
    metrics = evaluate_humanoid(env, model)
    
    # DataFrame para análisis
    df = pd.DataFrame(metrics)
    
    ArreglaryGuardar_df(df, model_path[:-4])

['sac_humanoid_final_1000000_20260430.zip', 'sac_humanoid_final_1000000_20260515_velocity_2_g0_95t0_005.zip', 'sac_humanoid_final_20000000_20260501.zip', 'sac_humanoid_final_20000000_20260505_velocity.zip', 'sac_humanoid_final_20000000_20260511_basic.zip', 'sac_humanoid_final_20000000_20260512_velocity.zip', 'sac_humanoid_final_20000000_20260516_knees_8_g95t5.zip', 'sac_humanoid_final_2000000_20260505_velocity.zip', 'sac_humanoid_final_30000000_20260520_knees_8_g95t5.zip', 'sac_humanoid_final_30000000_20260521_knees_8_g95t5.zip', 'sac_humanoid_final_30000000_20260525_knees_8_g90t5.zip', 'sac_humanoid_final_40000000_20260518_knees_8_g95t5.zip']
Cargando:  sac_humanoid_final_1000000_20260430.zip
step: 0
Cargando:  sac_humanoid_final_1000000_20260515_velocity_2_g0_95t0_005.zip
step: 1
Cargando:  sac_humanoid_final_20000000_20260501.zip
step: 2
Cargando:  sac_humanoid_final_20000000_20260505_velocity.zip
step: 3
Cargando:  sac_humanoid_final_20000000_20260511_basic.zip
step: 4
Cargando:  s